In [ ]:
# Load predictions from outputs directory
# predictions = np.load('../outputs/predictions/results.npy')
# plot_point_cloud(predictions[:, :3], labels=predictions[:, 3], title='Model Predictions')

## Prediction Results

Visualize model predictions on test data.

In [ ]:
def plot_point_cloud(points, labels=None, title='Point Cloud'):
    """
    Plot 3D point cloud.
    
    Args:
        points: Array of shape (N, 3) with x, y, z coordinates
        labels: Optional array of labels for coloring
        title: Title for the plot
    """
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    if labels is None:
        ax.scatter(points[:, 0], points[:, 1], points[:, 2], c='blue', s=1)
    else:
        ax.scatter(points[:, 0], points[:, 1], points[:, 2], c=labels, s=1, cmap='viridis')
    
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title(title)
    plt.show()

# Example usage:
# sample_points = np.random.randn(1000, 3)
# plot_point_cloud(sample_points)

## 3D Point Cloud Visualization

Visualize 3D point clouds from LiDAR data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import sys

# Add src directory to path
sys.path.insert(0, '../src')

# Visualization

Visualization of RiDAR_net predictions and point cloud analysis.

In [ ]:
#!/usr/bin/env python3
"""
Site1_10 visualisation (y in [7184396, 7184426]):

- LAS points in grey.
- Ground-truth pothole bboxes in green with labels (P13, P16, ...).
- Detected pothole locations (location_id; >=70% correct pothole patches) in red,
  as small rectangles around their centers (computed from cluster x0,y0).

Inputs:
- Site1_10/lidar/Site1_10.las
- Site1_10/label/Site1_10.csv
- Site1_10_test_multiscale.csv
- Input Files/Site1_10_RandLA-KPConv-Detection-perpatch_Site1_10.csv

Output:
- Site1_10_detected_vs_gt_2d_71844xx.png
"""

import laspy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
from adjustText import adjust_text  # pip install adjustText

LAS_PATH   = "Site1_10/lidar/Site1_10.las"
CSV_BBOX   = "Site1_10/label/Site1_10.csv"
CSV_CLUST  = "Site1_10_test_multiscale.csv"
CSV_PATCH  = "Input Files/Site1_10_RandLA-KPConv-Detection-perpatch_Site1_10.csv"

OUT_IMG    = "Site1_10_detected_vs_gt_2d_71844xx.png"

Y_MIN, Y_MAX = 7184396.0, 7184426.0
POTHOLE_CLASS = 1
THRESH_FRAC = 0.7  # >=70% correct pothole patches per location

def pick(colnames, candidates):
    for c in candidates:
        if c in colnames:
            return c
    raise KeyError(f"None of {candidates} found.")

def main():
    # ---------- Load LAS and crop to y-strip ----------
    print("Loading LAS...")
    las = laspy.read(LAS_PATH)
    x_all = np.asarray(las.x)
    y_all = np.asarray(las.y)

    mask_center = (y_all >= Y_MIN) & (y_all <= Y_MAX)
    x_bg = x_all[mask_center]
    y_bg = y_all[mask_center]
    print("Background points in strip:", x_bg.size)

    # ---------- Load cluster CSV ----------
    print("Loading cluster CSV...")
    df_clust = pd.read_csv(CSV_CLUST, low_memory=False)

    if "location_id" not in df_clust.columns:
        raise KeyError("location_id column not found in Site1_10_test_multiscale.csv")

    df_clust["damage_type_str"] = df_clust["damage_type"].astype(str).str.strip().str.lower()

    # ---------- Load per-patch predictions ----------
    print("Loading per-patch predictions...")
    df_patch = pd.read_csv(CSV_PATCH, low_memory=False)

    df = df_clust.merge(
        df_patch[["cluster_id", "true_class", "pred_class"]],
        on="cluster_id",
        how="inner",
    )

    df_pot = df[df["damage_type_str"] == "p"].copy()

    # ---------- Per-location voting ----------
    loc_rows = []
    for loc_id, g in df_pot.groupby("location_id"):
        n_patches = len(g)
        mask_correct = (g["true_class"] == POTHOLE_CLASS) & (g["pred_class"] == POTHOLE_CLASS)
        n_correct = int(mask_correct.sum())
        frac_correct = n_correct / float(n_patches)
        loc_rows.append({
            "location_id": loc_id,
            "n_patches": n_patches,
            "n_correct": n_correct,
            "frac_correct": frac_correct,
        })

    df_loc = pd.DataFrame(loc_rows)
    print("Total pothole locations (from clusters):", len(df_loc))

    df_loc_yes = df_loc[df_loc["frac_correct"] >= THRESH_FRAC].copy()
    print("Detected pothole locations with >=70% correct:", len(df_loc_yes))
    yes_locs = set(df_loc_yes["location_id"])

    # ---------- Compute centers of detected locations ----------
    x_cols = sorted([c for c in df_clust.columns if re.match(r"^x\d+$", c)])
    y_cols = sorted([c for c in df_clust.columns if re.match(r"^y\d+$", c)])
    if not x_cols or not y_cols:
        raise KeyError("No x* or y* columns found in Site1_10_test_multiscale.csv")

    x_col0 = x_cols[0]
    y_col0 = y_cols[0]
    print(f"Using {x_col0},{y_col0} as cluster-level point columns for centers")

    df_centers = (
        df_clust[df_clust["location_id"].isin(yes_locs)]
        .groupby("location_id")[[x_col0, y_col0]]
        .mean()
        .reset_index()
        .rename(columns={x_col0: "x_center", y_col0: "y_center"})
    )
    print("Centers computed for detected locations:", df_centers.shape[0])

    df_centers_strip = df_centers[
        (df_centers["y_center"] >= Y_MIN) &
        (df_centers["y_center"] <= Y_MAX)
    ].copy()
    print("Detected centers in strip:", len(df_centers_strip))

    def loc_to_label(loc_id):
        s = str(loc_id)
        parts = s.split("_")
        for i in range(len(parts) - 1):
            if parts[i].lower() == "p" and parts[i+1].isdigit():
                return f"P{parts[i+1]}"
        for token in parts:
            if token.lower().startswith("p") and token[1:].isdigit():
                return token.upper()
        return s

    df_centers_strip["label"] = df_centers_strip["location_id"].apply(loc_to_label)

    # ---------- Load ground-truth bboxes ----------
    print("Loading bbox CSV...")
    df_bbox = pd.read_csv(CSV_BBOX)
    df_bbox.columns = [c.strip().lower() for c in df_bbox.columns]

    col_damage_id = pick(df_bbox.columns, ["damage_id", "damageid", "id", "damage"])
    col_type      = pick(df_bbox.columns, ["damage_type", "damage", "type"])
    col_xmin      = pick(df_bbox.columns, ["xmin", "x_min", "x1", "xmin(m)"])
    col_xmax      = pick(df_bbox.columns, ["xmax", "x_max", "x2"])
    col_ymin      = pick(df_bbox.columns, ["ymin", "y_min", "y1"])
    col_ymax      = pick(df_bbox.columns, ["ymax", "y_max", "y2"])

    gt_boxes = []
    for _, row in df_bbox.iterrows():
        dt = str(row[col_type]).strip().lower()
        if dt != "p":
            continue

        xmin = float(row[col_xmin])
        xmax = float(row[col_xmax])
        ymin = float(row[col_ymin])
        ymax = float(row[col_ymax])

        if ymax < Y_MIN or ymin > Y_MAX:
            continue

        damage_id = str(row[col_damage_id]).strip()
        label = damage_id
        for token in damage_id.split("_"):
            if token.lower().startswith("p") and token[1:].isdigit():
                label = token.upper()
                break

        gt_boxes.append((xmin, ymin, xmax, ymax, label))

    print("Ground-truth pothole boxes in strip:", len(gt_boxes))

    # ---------- Plot ----------
    print("Plotting...")
    plt.figure(figsize=(10, 6), dpi=400)

    # lighter, more transparent background
    plt.scatter(
        x_bg, y_bg,
        s=0.2,
        c="darkgray",
        marker=".",
        linewidths=0,
        alpha=0.2
    )

    ax = plt.gca()

    gt_texts = []
    for (xmin, ymin, xmax, ymax, label) in gt_boxes:
        w = xmax - xmin
        h = ymax - ymin
        rect = plt.Rectangle(
            (xmin, ymin), w, h,
            fill=False,
            edgecolor="green",
            linewidth=0.5,
            alpha=0.9
        )
        ax.add_patch(rect)
        txt = ax.text(
            xmin,
            ymax + 0.1,
            label,
            color="green",
            fontsize=5,
            weight="bold",
            ha="left",
            va="bottom"
        )
        gt_texts.append(txt)

    # Detected centers in red
    half_size = 0.3
    det_texts = []
    for _, row in df_centers_strip.iterrows():
        xc = row["x_center"]
        yc = row["y_center"]
        label = row["label"]
        rect = plt.Rectangle(
            (xc - half_size, yc - half_size),
            2 * half_size, 2 * half_size,
            fill=False,
            edgecolor="red",
            linewidth=0.7,
            alpha=0.9
        )
        ax.add_patch(rect)
        txt = ax.text(
            xc,
            yc + half_size + 0.1,
            label,
            color="red",
            fontsize=5,
            weight="bold",
            ha="center",
            va="bottom"
        )
        det_texts.append(txt)

    # de-overlap labels
    adjust_text(
        gt_texts + det_texts,
        ax=ax,
        only_move={'points': 'y', 'texts': 'xy'},
        arrowprops=dict(arrowstyle="-", color="black", lw=0.3, alpha=0.5)
    )

    ax.set_aspect("equal", adjustable="box")
    plt.xlabel("X")
    plt.ylabel("Y")
    plt.title(f"Site1_10: Ground Truth (green) and Detected (red), y in [{Y_MIN}, {Y_MAX}]")
    plt.xlim(x_bg.min(), x_bg.max())
    plt.ylim(Y_MIN, Y_MAX)

    plt.tight_layout()
    plt.savefig(OUT_IMG, dpi=400)
    plt.close()
    print(f"Saved high-res comparison to {OUT_IMG}")

if __name__ == "__main__":
    main()
